# Task 2 — station selection and waveform processing

`waveforms.fetch_and_process` does everything in this task. The rules and
their sources are in the module docstring and `auto_tdmt.cfg` §2:
distance window by magnitude; unusable data rejected; per-component SNR;
amplitude outliers; azimuth-balanced round-robin selection.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
# work in a scratch archive so the real events/ are untouched
os.environ.setdefault("AUTO_TDMT_EVENTS", str(Path.home() / "work" / "proj_tdmt_NZ" / "notebook_runs"))
import config
from config import P            # every tunable, from auto_tdmt.cfg
EVENT = "2026p669681"
print("parameters from", P.source)

In [ ]:
print(json.dumps(P.as_dict()["station"], indent=2))

## 2.1 Run the task on one event and one band

In [ ]:
import waveforms
from geonet import get_event
ev = get_event(EVENT)
band = config.band_candidates(ev.prelim_mag)[0]      # first band of the menu
wd = config.EVENTS_DIR / ev.public_id / "task2"
pool, dropped = waveforms.fetch_and_process(ev, wd, band)
print(f"band {config.band_tag(band)}: {len(pool)} selected, {len(dropped)} not")

## 2.2 What was selected, and why the rest were not

In [ ]:
print(f"{'station':14s} {'dist':>5s} {'az':>4s} sec {'SNR Z/R/T':>17s} {'med':>5s} {'win':>4s} {'amp':>5s}")
for r in pool:
    print(f"{waveforms.station_id(r):14s} {r['distance_km']:5.0f} {r['azimuth']:4.0f} "
          f"{r['sector']:3d} {r['snr']['Z']:5.1f}/{r['snr']['R']:5.1f}/{r['snr']['T']:5.1f} "
          f"{r['snr_med']:5.1f} {r['window_end_s']:4d} {r.get('amp_ratio', float('nan')):5.2f}")
print()
for d in dropped:
    print(f"{d['station']:14s} {d['reason']}")

## 2.3 The traces the inversion will see

In [ ]:
from obspy import read
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(pool), 1, figsize=(10, 1.2 * len(pool)), sharex=True)
for ax, r in zip(axes, pool):
    for comp, c in zip("ZRT", ("k", "#0072B2", "#E69F00")):
        tr = read(str(wd / f"{waveforms.station_id(r)}.{comp}.dat"), format="SAC")[0]
        t = tr.times() + tr.stats.sac.b
        ax.plot(t, tr.data, c, lw=0.6, label=comp)
    ax.axvline(r["window_end_s"], color="0.5", ls="--", lw=0.8)
    ax.set_ylabel(r["station"], rotation=0, ha="right", fontsize=8)
    ax.set_yticks([])
axes[0].legend(ncol=3, fontsize=7, loc="upper right")
axes[-1].set_xlabel("s after origin  (dashed = end of the inverted window)")
plt.tight_layout(); plt.show()

## What to check
- Does the SNR threshold agree with your eye? (`station.snrMin`)
- Is the inverted window (dashed) long enough for the far stations?
  (`station.maxWindowS`, `groupVelKms`, `windowTailS`)
- Did the round-robin leave out a station you would have kept?
  (`station.maxStations`)